In [7]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [8]:
#kütüphaneleri yükleme
import pandas as pd


In [ ]:
df = pd.read_csv(r"C:\Users\hazar\Desktop\kodlama\python kodlama pratikleri\practise\demografik_risk_profilleme\dataset\insurance.csv")

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (381712828.py, line 1)

In [ ]:
df.head() # ilk 5 satırı gösterir


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [ ]:
df.shape # satır ve sütun sayısını gösterir

(1338, 7)

In [ ]:
df.info() # veri seti hakkında bilgi verir

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [ ]:
df.isnull().sum() # eksik değerleri gösterir

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

## 2 - Veri Doğrulama, Tekillik ve İstatistiksel Özet

In [ ]:
# Tekrar eden (Duplicate) satırları kontrol etme
df.duplicated().sum() # tekrar eden satır sayısını gösterir

np.int64(1)

In [ ]:
# tekrar kontrolü sıfırdan büyük (1) çıktığı için tekrar eden satırları silme işlemi yapıyoruz
df[df.duplicated(keep=False)] # tekrar eden satırları gösterir

,age,sex,bmi,children,smoker,region,charges
195,19,male,30.59,0,no,northwest,1639.5631
581,19,male,30.59,0,no,northwest,1639.5631


In [ ]:
df = df.drop_duplicates().reset_index(drop=True) # tekrar eden satırları siler ve indexleri resetler

In [ ]:
print("Veri seti temizlendi. Tekrar eden satırlar silindi ve indexler resetlendi.", df.shape[0])

Veri seti temizlendi. Tekrar eden satırlar silindi ve indexler resetlendi. 1337


In [ ]:
# verideki sayısal sütunların istatistiksel özetini gösterme
df.describe().T # sayısal sütunların istatistiksel özetini gösterir

,count,mean,std,min,25%,50%,75%,max
age,1337.0,39.222139,14.044333,18.0000,27.000,39.0000,51.00000,64.00000
bmi,1337.0,30.663452,6.100468,15.9600,26.290,30.4000,34.70000,53.13000
children,1337.0,1.095737,1.205571,0.0000,0.000,1.0000,2.00000,5.00000
charges,1337.0,13279.121487,12110.359656,1121.8739,4746.344,9386.1613,16657.71745,63770.42801


In [ ]:
# verinin istatistiksel özetini inceledikten sonra sınıflandırma ve gerekli algoritmasının yazıyoruz

class DemographicRiskProfiling:
    def __init__(self, dataframe):
        self.df = dataframe

    def bmi_category(self,bmi):
        #bmi kategorilerini belirleme
        if bmi < 18.5:
            return "zayıf"
        elif 18.5 < bmi <25.0:
            return "normal"
        elif 25.0 < bmi < 30.0:
            return "fazla kilolu"
        else: 
            return "obez" 
        
    def portfoy_etiketle(self):
        #bmi kategorilerini dataframe'e ekleme
        self.df["bmi_category"] = self.df["bmi"].apply(self.bmi_category)
        return self.df["bmi_category"].value_counts()

    def risk_carpani_hesapla(self,bmi,smoker,age):
        #Müşterinin bmi, sigara içme durumu ve yaşına göre risk çarpanını hesaplama

        #1 bmi tabanlı temel risk çarpanı
        if bmi < 18.5:
            temel_carpan = 1.1
        elif bmi < 25.0:
            temel_carpan = 1.0
        elif bmi < 30.0:
            temel_carpan = 1.25
        else: 
            temel_carpan = 1.5

        #2 sigara içme durumu tabanlı risk çarpanı
        if smoker == "yes":
            sigara_carpani = 1.80
        else:
            sigara_carpani = 1.0

        #3 yaş tabanlı risk çarpanı
        yas_carpani = 1.0 + (age/100)  # yaşın %1'i kadar risk çarpanı eklenir

        #Toplam aktüeryal risk çarpanı
        toplam_risk_carpani = temel_carpan * sigara_carpani * yas_carpani
        return round(toplam_risk_carpani, 2)


    def portfoy_fiyatlandır(self):
        # her müşterinin risk çarpanına göre portföy fiyatlandırması yapma
        self.df["risk_carpani"] = self.df.apply(lambda x: self.risk_carpani_hesapla(x["bmi"], x["smoker"], x["age"]), axis=1)
        self.df["portfoy_fiyat"] = self.df["charges"] * self.df["risk_carpani"]
        return self.df[["age", "bmi", "smoker", "charges", "risk_carpani", "portfoy_fiyat"]].head(10)  # ilk 10 satırı gösterir


    def canli_müsteri_sorgula(self):
        # canlı müşteri sorgulama fonksiyonu
       print("\n" + "="+50)
       print("Aktüeryal Canlı Risk Sorgulama Ekranı")
       print("="+50)

       ad = input("Müşteri Adı: ")
       soyad = input("Müşteri Soyadı: ")
       yas = int(input("Müşteri Yaşı: "))
       boy_cm = float(input("Boy (cm): "))
       kilo_kg = float(input("Kilo (kg): "))
       sigara = input("Sigara İçiyor mu? (yes/no): ").strip().lower()

       # temel hesaplamalar
       boy_m = boy_cm / 100.0
       bmi = kilo_kg / (boy_m**2)

       #bmi kategorisini belirleme
       bmi_kategori = self.bmi_category(bmi)

       # toplam risk çarpanını hesaplama
       risk_carpani = self.risk_carpani_hesapla(bmi, sigara, yas)

       taban_prim = 5000.0
       netteklif = taban_prim * risk_carpani

       #sonuçları ekrana yazdırma
       print(f"\nMüşteri: {ad} {soyad}")
       print(f"Yaş: {yas}")
       print(f"Boy: {boy_cm} cm")
       print(f"Kilo: {kilo_kg} kg")
       print(f"BMI: {bmi:.2f}")
       print(f"BMI Kategorisi: {bmi_kategori}")
       print(f"Sigara İçme Durumu: {sigara}")
       print(f"Toplam Risk Çarpanı: {risk_carpani}")
       print(f"Net Teklif: {netteklif:.2f} TL")


In [ ]:
# sınıflandırma algoritmasını çalıştırma
profiler = DemographicRiskProfiling(df)

In [ ]:
dagilim = profiler.portfoy_etiketle()
print(dagilim)

bmi_category
obez            709
fazla kilolu    384
normal          224
zayıf            20
Name: count, dtype: int64


In [ ]:
fişyatlandırma = profiler.portfoy_fiyatlandır()
print(fişyatlandırma)

   age     bmi smoker      charges  risk_carpani  portfoy_fiyat
0   19  27.900    yes  16884.92400          2.68   45251.596320
1   18  33.770     no   1725.55230          1.77    3054.227571
2   28  33.000     no   4449.46200          1.92    8542.967040
3   33  22.705     no  21984.47061          1.33   29239.345911
4   32  28.880     no   3866.85520          1.65    6380.311080
5   31  25.740     no   3756.62160          1.64    6160.859424
6   46  33.440     no   8240.58960          2.19   18046.891224
7   37  27.740     no   7281.50560          1.71   12451.374576
8   37  29.830     no   6406.41070          1.71   10954.962297
9   60  25.840     no  28923.13692          2.00   57846.273840


In [ ]:
canlı_müşteri = profiler.canli_müsteri_sorgula()
print(canlı_müşteri)

NameError: name 'profiler' is not defined